# Prepare and audit MUFASA author-year citation metadata

This notebook checks the source-paper citation that will be rendered as `(Njoku et al., 2022)`-style metadata in SFT answers. It is an audit, not a paper or pair filter.

The default run selects 30 papers by a deterministic hash-random ordering across the complete frozen corpus. It writes only a preview under `citation_audits/`. The canonical `citation_metadata.parquet` is mechanically unreachable until `FULL_RUN=True`. No language model or API is used.

In [8]:
# =============================== controls ==================================
import hashlib
import importlib
import os
import re
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    drive = None
else:
    drive.mount("/content/drive")

# On Colab, edit this one path only if your MUFASA folder is elsewhere.
DRIVE_DATA = Path("/content/drive/MyDrive/MUFASA/01-data-engineering/data-extraction")
override = os.environ.get("MUFASA_DATA_DIR", "").strip()
candidates = ([Path(override)] if override else []) + ([DRIVE_DATA] if drive else [])
candidates += [
    folder for start in [Path.cwd(), *Path.cwd().parents]
    for folder in (start, start / "01-data-engineering" / "data-extraction")
]
DATA = next((folder.resolve() for folder in candidates
             if (folder / "corpus_splits" / "manifest.parquet").is_file()), None)
if DATA is None:
    raise FileNotFoundError("Set DRIVE_DATA or MUFASA_DATA_DIR to the data-extraction folder")
for module in ("mufasa_citations.py", "mufasa_dataset.py"):
    if not (DATA / module).is_file():
        raise FileNotFoundError(f"Place {module} beside this notebook at {DATA}")
sys.path.insert(0, str(DATA))

import mufasa_citations as citations
import mufasa_dataset as funnel
importlib.reload(citations)
importlib.reload(funnel)

FULL_RUN = True          # Only True may write canonical citation_metadata.parquet.
SAMPLE_SIZE = 30          # Used only while FULL_RUN is False.
SEED = 7
AUDIT_MODE = citations.AUDIT_MODE

SPLIT_MANIFEST = DATA / "corpus_splits" / "manifest.parquet"
DOCUMENTS = DATA / "mufasa_corpus" / "manifests" / "documents.parquet"
AUTHORS_CACHE = DATA / "production" / "authors_cache.parquet"
MARKDOWN_ROOT = DATA / "mufasa_corpus" / "parsed" / "markdown"
CANONICAL_PATH = DATA / "citation_metadata.parquet"
AUDIT_DIR = DATA / "citation_audits"
safe_version = re.sub(r"[^a-zA-Z0-9_.-]+", "-", citations.CITATION_METADATA_VERSION)
PREVIEW_PATH = AUDIT_DIR / f"preview-seed{SEED}-n{SAMPLE_SIZE}-{safe_version}.parquet"
OUTPUT_PATH = CANONICAL_PATH if FULL_RUN else PREVIEW_PATH

if SAMPLE_SIZE < 1:
    raise ValueError("SAMPLE_SIZE must be positive")
for required in (SPLIT_MANIFEST, DOCUMENTS, AUTHORS_CACHE):
    if not required.is_file():
        raise FileNotFoundError(required)
if not MARKDOWN_ROOT.is_dir():
    raise FileNotFoundError(MARKDOWN_ROOT)
if not FULL_RUN and OUTPUT_PATH.resolve() == CANONICAL_PATH.resolve():
    raise RuntimeError("Preview mode is forbidden from writing canonical citation metadata")

split_values = pq.read_table(SPLIT_MANIFEST, columns=["paper_id"])["paper_id"].to_pylist()
ALL_PAPER_IDS = sorted({
    paper_id for value in split_values
    if (paper_id := citations.short_openalex_id(value))
})
HASH_ORDER = sorted(
    ALL_PAPER_IDS,
    key=lambda paper_id: hashlib.sha256(f"{SEED}:{paper_id}".encode("utf-8")).hexdigest(),
)
SELECTED_PAPER_IDS = None if FULL_RUN else HASH_ORDER[:min(SAMPLE_SIZE, len(HASH_ORDER))]

print("data root       :", DATA)
print("audit mode      :", AUDIT_MODE)
print("frozen papers   :", f"{len(ALL_PAPER_IDS):,}")
print("requested papers:", "FULL CORPUS" if FULL_RUN else len(SELECTED_PAPER_IDS))
print("output          :", OUTPUT_PATH)
print("canonical write :", "ENABLED" if FULL_RUN else "BLOCKED")


data root       : C:\CodingWorld\Hackathons\AfricanDeepTechChallenge\MUFASA\01-data-engineering\data-extraction
audit mode      : AUDIT_ONLY
frozen papers   : 10,480
requested papers: FULL CORPUS
output          : C:\CodingWorld\Hackathons\AfricanDeepTechChallenge\MUFASA\01-data-engineering\data-extraction\citation_metadata.parquet
canonical write : ENABLED


In [9]:
# ======================= run the metadata audit =============================
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
CITATION_AUDIT = citations.prepare_citation_metadata(
    split_manifest=SPLIT_MANIFEST,
    documents_path=DOCUMENTS,
    authors_cache_path=AUTHORS_CACHE,
    markdown_root=MARKDOWN_ROOT,
    paper_ids=SELECTED_PAPER_IDS,
    output_path=OUTPUT_PATH,
)

expected_rows = len(ALL_PAPER_IDS) if FULL_RUN else len(SELECTED_PAPER_IDS)
if len(CITATION_AUDIT) != expected_rows:
    raise RuntimeError(f"Expected {expected_rows:,} citation rows, got {len(CITATION_AUDIT):,}")
if not OUTPUT_PATH.is_file():
    raise RuntimeError(f"Citation audit was not written: {OUTPUT_PATH}")
if not FULL_RUN:
    observed = set(CITATION_AUDIT["paper_id"])
    if observed != set(SELECTED_PAPER_IDS):
        raise RuntimeError("Preview rows do not match the deterministic frozen-corpus sample")

print(f"Completed {AUDIT_MODE}: {len(CITATION_AUDIT):,} papers")
print("written:", OUTPUT_PATH)
print("This audit removed zero papers and zero training pairs.")
if not FULL_RUN:
    print("PREVIEW ONLY: citation_metadata.parquet was not written.")


citation audit (10,480 papers):   0%|          | 0/10480 [00:00<?, ?paper/s]

Completed AUDIT_ONLY: 10,480 papers
written: C:\CodingWorld\Hackathons\AfricanDeepTechChallenge\MUFASA\01-data-engineering\data-extraction\citation_metadata.parquet
This audit removed zero papers and zero training pairs.


In [10]:
# ============================= summary table ================================
summary = citations.audit_summary(CITATION_AUDIT)
status_counts = CITATION_AUDIT["citation_status"].value_counts(dropna=False)
summary_rows = [
    {"metric": "papers audited", "count": len(CITATION_AUDIT),
     "meaning": "deterministic sample" if not FULL_RUN else "complete frozen corpus"},
    *[
        {"metric": f"citation_status = {status}", "count": int(count),
         "meaning": "diagnostic classification; not a training-data gate"}
        for status, count in status_counts.items()
    ],
    {"metric": "fallback labels", "count": int(summary["fallback_rows"]),
     "meaning": "author/year could not form a normal author-year label"},
    {"metric": "ambiguous author-year keys",
     "count": int(summary["ambiguous_author_year_rows"]),
     "meaning": "same first-author/year key appears more than once in this run"},
]
SUMMARY_TABLE = pd.DataFrame(summary_rows)
display(SUMMARY_TABLE)

component_counts = pd.concat(
    [CITATION_AUDIT["author_status"].value_counts().rename("author"),
     CITATION_AUDIT["year_status"].value_counts().rename("year")],
    axis=1,
).fillna(0).astype(int).rename_axis("status").reset_index()
display(component_counts)


,metric,count,meaning
0,papers audited,10480,complete frozen corpus
1,citation_status = VERIFIED_DOCUMENT,4837,diagnostic classification; not a training-data...
2,citation_status = METADATA_ONLY,2436,diagnostic classification; not a training-data...
3,citation_status = CONFLICT,2006,diagnostic classification; not a training-data...
4,citation_status = CORRECTED_DOCUMENT,1201,diagnostic classification; not a training-data...
5,fallback labels,0,author/year could not form a normal author-yea...
6,ambiguous author-year keys,2707,same first-author/year key appears more than o...


,status,author,year
0,VERIFIED_DOCUMENT,8545,5964
1,CORRECTED_DOCUMENT,1209,222
2,METADATA_ONLY,476,2471
3,CONFLICT,250,1823


In [11]:
# ======================== document-corrected samples ========================
corrected = CITATION_AUDIT[
    (CITATION_AUDIT["author_status"] == citations.STATUS_CORRECTED)
    | (CITATION_AUDIT["year_status"] == citations.STATUS_CORRECTED)
].copy()
corrected_columns = [
    "paper_id", "title", "openalex_label", "citation_label",
    "document_first_author_family", "document_year",
    "author_status", "year_status", "author_evidence", "year_evidence",
]
print(f"Document-corrected rows: {len(corrected):,}")
if corrected.empty:
    display(pd.DataFrame({"result": ["No corrected row occurred in this random sample."]}))
else:
    with pd.option_context("display.max_colwidth", 180, "display.max_columns", None):
        display(corrected[corrected_columns].head(10).reset_index(drop=True))


Document-corrected rows: 1,411


,paper_id,title,openalex_label,citation_label,document_first_author_family,document_year,author_status,year_status,author_evidence,year_evidence
0,W108716228,"Prevalence and effect of schistosome and soil-transmitted helminth infection on labour input in rice-growing communities of Ogun State, Nigeria","Sam-Wobo et al., 2022","Sam-Wobo et al., 2013",Sam-Wobo,2013,VERIFIED_DOCUMENT,CORRECTED_DOCUMENT,Sammy Olufemi Sam-Wobo,"Epidemiology Biostatistics and Public Health - 2013, Volume 10, Number 2"
1,W1504074386,Assessment of effects of controlled land use types on soil quality using inferential method,"e et al., 2009","T et al., 2009",T,2009,CORRECTED_DOCUMENT,VERIFIED_DOCUMENT,Assessment of effects of controlled land use types on soil quality using inferential method,"African Journal of Biotechnology Vol. 8 (22), pp. 6267-6271, 16 November, 2009 Available online at [http://www.academicjournals.org/AJB](http://www.academicjournals.org/AJB) IS..."
2,W1516835331,Prevalence of malnutrition among HIV-infected children in Central and West-African HIV-care programmes supported by the Growing Up Programme in 2011: a cross-sectional study,"Group et al., 2015","Jesson et al., 2015",Jesson,2015,CORRECTED_DOCUMENT,VERIFIED_DOCUMENT,Jesson et al. BMC Infectious Diseases (2015) 15:216 DOI 10.1186/s12879-015-0952-6,© 2015 Jesson et al.; licensee BioMed Central. This is an Open Access article distributed under the terms of the Creative Commons Attribution License ([http://creativecommons.o...
3,W1524462659,Phytochemical And Antimicrobial Effects Of &lt;i&gt;Chrozophora senegalensis&lt;/i&gt;,"Usman et al., 2008","Usman et al., 2007",Usman,2007,VERIFIED_DOCUMENT,CORRECTED_DOCUMENT,"Usman et al., Afr. J. Trad. CAM (2007) 4 (4): 488 – 494",ISSN 0189-6016©2007
4,W1544107599,Comparative Studies on the Physicochemical and Sensory Properties of Watermelon (Citrullus lanatus) and Melon (Citrullus vulgaris) Seed Flours Used in “EGUSI” Soup Preparation,"Akusu & Kiin-Kabari, 2015","Kiin-Kabari & Akusu, 2015",Kiin-Kabari,,CORRECTED_DOCUMENT,CONFLICT,"Monday O. Akusu¹ & David B. Kiin-Kabari¹ 1 Department of Food Science and Technology, Rivers State University of Science and Technology, Port Harcourt, Nigeria","Journal of Food Research; Vol. 4, No. 5; 2015 ISSN 1927-0887 E-ISSN 1927-0895 u Published by Canadian Center of Science and Education /u | Journal of Food Research; Vol. 4, No...."
5,W1572154625,An improved dual sensor summation method with application to four-component (4-C) seafloor seismic data from the Niger Delta,"Ogagarue & Ebeniro, 2015","Ebeniro & Ogagarue, 2015",Ebeniro,,CORRECTED_DOCUMENT,CONFLICT,Difference O. Ogagarue¹ & Joseph O. Ebeniro 2,"Earth Science Research; Vol. 4, No. 2; 2015 ISSN 1927-0542 E-ISSN 1927-0550 u Published by Canadian Center of Science and Education /u | Earth Science Research; Vol. 4, No. 2; ..."
6,W1573658597,"Aquifer characteristics and groundwater recharge pattern in a typical basement complex, Southwestern Nigeria","Su & B Olatinsu, 2010","B Olatinsu & Su, 2010",B Olatinsu,,CORRECTED_DOCUMENT,CONFLICT,B. S. Badmus and O. B. Olatinsu,"African Journal of Environmental Science and Technology Vol. 4 (6), pp. 328-342, June, 2010 Available online at [http://www.academicjournals.org/AJEST](http://www.academicjourn..."
7,W1574336839,"Astrocyte morphology, heterogeneity, and density in the developing African giant rat (Cricetomys gambianus)","Olude et al., 2015","Olopade et al., 2015",Olopade,2015,CORRECTED_DOCUMENT,VERIFIED_DOCUMENT,"Edited by: Yun-Qing Li, The Fourth Military Medical University, China Reviewed by: James C. Vickers, University of Tasmania, Australia Anne-Karine Bouzier-Sore, Centre National...","Received: 09 March 2015 Accepted: 11 May 2015 Published: 26 May 2015 Citation: Olude MA, Mustapha OA, Aderounmu OA, Olopade JO and Ihunwo AO (2015) Astrocyte morphology, hetero..."
8,W1584767745,"Comparative Studies of Steel, Bamboo and Rattan as Reinforcing Bars in Concrete: Tensile and Flexural Characteristics

In [12]:
# ============================== conflict samples ============================
conflicts = CITATION_AUDIT[
    (CITATION_AUDIT["citation_status"] == citations.STATUS_CONFLICT)
    | (CITATION_AUDIT["author_status"] == citations.STATUS_CONFLICT)
    | (CITATION_AUDIT["year_status"] == citations.STATUS_CONFLICT)
].copy()
conflict_columns = [
    "paper_id", "title", "openalex_first_author_family",
    "document_first_author_family", "openalex_year", "document_year",
    "citation_label", "author_status", "year_status",
    "author_evidence", "year_evidence",
]
print(f"Conflict rows: {len(conflicts):,}")
if conflicts.empty:
    display(pd.DataFrame({"result": ["No conflict occurred in this random sample."]}))
else:
    with pd.option_context("display.max_colwidth", 180, "display.max_columns", None):
        display(conflicts[conflict_columns].head(10).reset_index(drop=True))


Conflict rows: 2,006


,paper_id,title,openalex_first_author_family,document_first_author_family,openalex_year,document_year,citation_label,author_status,year_status,author_evidence,year_evidence
0,W1263080160,"Hydrochemistry of surface water and groundwater in the shale bedrock, Cross River Basin and Niger Delta Region, Nigeria",Nganje,Nganje,2015,,"Nganje et al., 2015",VERIFIED_DOCUMENT,CONFLICT,T. N. Nganje,Received: 30 December 2014 / Accepted: 23 June 2015 / Published online: 16 July 2015 The Author(s) 2015. This article is published with open access at Springerlink.com | Receiv...
1,W1481448257,"Assessment of Heavy Metals Pollution in Soils and Vegetation around Selected Industries in Lagos State, Nigeria",Adesuyi,Adesuyi1,2015,2015,"Adesuyi et al., 2015",CONFLICT,VERIFIED_DOCUMENT,"Adeola Alex Adesuyi1 , Kelechi Longinus Njoku², Modupe Olatunde Akinola","How to cite this paper: Adesuyi, A.A., Njoku, K.L. and Akinola, M.O. (2015) Assessment of Heavy Metals Pollution in Soils and Vegetation around Selected Industries in Lagos Sta..."
2,W1483996455,Environmental Impact Analysis of the Emission from Petroleum Refineries in Nigeria,Oladimeji,Oladimeji,2015,,"Oladimeji et al., 2015",VERIFIED_DOCUMENT,CONFLICT,"Oladimeji T. E. 1, Sonibare J. A. 2, Odunfa K. M. 3 & Oresegun O. R. 1","Energy and Environment Research; Vol. 5, No. 1; 2015 ISSN 1927-0569 E-ISSN 1927-0577 u Published by Canadian Center of Science and Education /u | Energy and Environment Researc..."
3,W1493677654,Dietary Effects of Increasing Levels of Pigeon Pea Meal on Rabbit Performance,Akande,Akande,2015,,"Akande, 2015",VERIFIED_DOCUMENT,CONFLICT,"Kemi E. Akande¹ 1 Department of Animal Production, Faculty of Agriculture, Abubakar Tafawa Balewa University, Bauchi State, Nigeria","Journal of Agricultural Science; Vol. 7, No. 7; 2015 ISSN 1916-9752 E-ISSN 1916-9760 u Published by Canadian Center of Science and Education /u | Journal of Agricultural Scienc..."
4,W1494572909,"Seasonal variation in the physicochemistry of a small tropical reservoir (Aiba Reservoir, Iwo, Osun, Nigeria)",Atobatele,Atobatele,2008,,"Atobatele et al., 2008",VERIFIED_DOCUMENT,CONFLICT,"Atobatele, Oluwatosin Ebenezer and Ugwumba, O. Alex","African Journal of Biotechnology Vol. 7 (12), pp. 1962-1971, 17 June, 2008 Available online at [http://www.academicjournals.org/AJB](http://www.academicjournals.org/AJB) ISSN 1..."
5,W1538750661,"Assessment of Activity Concentration of Radionuclides in Sediment from Oil Producing Communities of Delta State, Nigeria",Iwetan,Iwetan1,2015,2015,"Iwetan et al., 2015",CONFLICT,VERIFIED_DOCUMENT,"Caroline Nihinlola Iwetan1 , Ibiyinka Agboola Fuwape¹, Adeseye Muyiwa Arogunjo¹","How to cite this paper: Iwetan, C.N., Fuwape, I.A., Arogunjo, A.M. and Obor, G. (2015) Assessment of Activity Concentra- tion of Radionuclides in Sediment from Oil Producing Co..."
6,W1542443322,Screening of crude extracts of six medicinal plants used in South-West Nigerian unorthodox medicine for anti-methicillin resistant Staphylococcus aureus activity,Akinyemi,Akinyemi,2005,,"Akinyemi et al., 2005",VERIFIED_DOCUMENT,CONFLICT,"Kabir O Akinyemi , Olukayode Oladapo, Chidi E Okwara, Christopher C Ibe and Kehinde A Fasure",Corresponding author Published: 11 March 2005 Received: 10 October 2004 | Corresponding author Published: 11 March 2005 Received: 10 October 2004
7,W1544107599,Comparative Studies on the Physicochemical and Sensory Properties of Watermelon (Citrullus lanatus) and Melon (Citrullus vulgaris) Seed Flours Used in “EGUSI” Soup Preparation,Akusu,Kiin-Kabari,2015,,"Kiin-Kabari & Akusu, 2015",CORRECTED_DOCUMENT,CONFLICT,"Monday O. Akusu¹ & David B. Kiin-Kabari¹ 1 Department of Food Science and Technology, Rivers State University of Science and Technology, Port Harcourt, Nigeria","Journal of Food Research; Vol. 4, No. 5; 2015 ISSN 1927-0887 E-ISSN 1927-0895 u Published by Canadian Center of Science and Education /u | Journal of Food Research; Vol. 4, No...."
8,W1545579013,"Exposure to Emissions from Keros

In [13]:
# ========================== metadata-only samples ===========================
metadata_only = CITATION_AUDIT[
    CITATION_AUDIT["citation_status"] == citations.STATUS_METADATA
].copy()
metadata_columns = [
    "paper_id", "title", "openalex_first_author", "openalex_year",
    "citation_label", "author_status", "year_status",
    "metadata_source", "fallback_used",
]
print(f"Metadata-only rows: {len(metadata_only):,}")
if metadata_only.empty:
    display(pd.DataFrame({"result": ["No metadata-only row occurred in this random sample."]}))
else:
    with pd.option_context("display.max_colwidth", 180, "display.max_columns", None):
        display(metadata_only[metadata_columns].head(10).reset_index(drop=True))


Metadata-only rows: 2,436


,paper_id,title,openalex_first_author,openalex_year,citation_label,author_status,year_status,metadata_source,fallback_used
0,W1503251855,Genotype x Environment Interaction and Yield-Stability Analyses of Rice Grown in Tropical Inland Swamp,Adesola L. Nassir,2011,"Nassir & Ariyo, 2011",VERIFIED_DOCUMENT,METADATA_ONLY,OPENALEX+DOCUMENT,False
1,W1507330190,Geotechnical Characterization of some Clayey Soils for Use as Landfill Liner,O. Ojuri Oluwapelumi,2015,"Oluwapelumi, 2015",VERIFIED_DOCUMENT,METADATA_ONLY,OPENALEX+DOCUMENT,False
2,W1512855654,Stem Bark Extracts of &lt;i&gt;Ficus exasperata&lt;/i&gt; protects the Liver against Paracetamol induced toxicity in Wistar Rats,Adaze Bijou Enogieru,2015,"Enogieru et al., 2015",VERIFIED_DOCUMENT,METADATA_ONLY,OPENALEX+DOCUMENT,False
3,W1513321141,"Cassava Intake and Vitamin A Status among Women and Preschool Children in Akwa-Ibom, Nigeria",Fabiana F. De Moura,2015,"De Moura et al., 2015",METADATA_ONLY,VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,False
4,W1516783031,Description of physiotherapy services in a mental health institution in Nigeria,Caleb Ademola Gbiri,2011,"Gbiri et al., 2011",VERIFIED_DOCUMENT,METADATA_ONLY,OPENALEX+DOCUMENT,False
5,W1529863835,Nutritional Quality Assessment of Complementary Foods produced from Fermented and Malted Quality Protein Maize Fortified with Soy Bean Flour,Sumbo H. Abiose,2015,"Abiose et al., 2015",METADATA_ONLY,VERIFIED_DOCUMENT,OPENALEX+DOCUMENT,False
6,W1539427703,"Antibiotic Susceptibility Pattern of Extended Spectrum Betalactamase (ESBL) Producers and other Bacterial Pathogens in Kano, Nigeria",Emmanuel Onwubiko Nwankwo,2015,"Nwankwo et al., 2015",VERIFIED_DOCUMENT,METADATA_ONLY,OPENALEX+DOCUMENT,False
7,W1543771015,Leaf Protein Electrophoresis and Taxonomy of Species of Jatropha L. (Euphorbiaceae),Olaniran Temitope Oladipo,2012,"Oladipo & Illoh, 2012",VERIFIED_DOCUMENT,METADATA_ONLY,OPENALEX+DOCUMENT,False
8,W1553907965,Seronegative conversion of an HIV positive subject treated with &lt;i&gt;Nigella sativa&lt;/i&gt; and honey,A.A. Onifade,2015,"Onifade et al., 2015",VERIFIED_DOCUMENT,METADATA_ONLY,OPENALEX+DOCUMENT,False
9,W1565008855,Fuzzy Based Design for Third-Party Pipeline Failures in the Niger Delta Region of Nigeria,GO Ariavie,2015,"Ariavie et al., 2015",VERIFIED_DOCUMENT,METADATA_ONLY,OPENALEX+DOCUMENT,False


In [14]:
# ====================== rendered citation-format preview ====================
# This exercises the exact low-level response formatter. The placeholder
# answer and basis are for display only; the training builder supplies the real ones.
rendered_rows = []
for row in CITATION_AUDIT.head(10).to_dict(orient="records"):
    trusted = row["citation_status"] in {
        citations.STATUS_VERIFIED, citations.STATUS_CORRECTED,
    }
    provenance = funnel.LEARNED_STUDY if trusted else funnel.UNVERIFIED_STUDY
    rendered = funnel.format_provenance_response(
        "[The factual or reasoning answer appears here.]",
        provenance=provenance,
        citation_label=row["citation_label"],
        study_basis="[The semantic study descriptor appears here.]",
    )
    rendered_rows.append({
        "paper_id": row["paper_id"],
        "citation_status": row["citation_status"],
        "rendered_assistant_turn": rendered,
    })
with pd.option_context("display.max_colwidth", 500, "display.max_columns", None):
    display(pd.DataFrame(rendered_rows))

print("Preview complete. This notebook applied no training-data gate.")
print("To prepare the canonical table, review these cells, set FULL_RUN=True, and Run all.")


,paper_id,citation_status,rendered_assistant_turn
0,W108716228,CORRECTED_DOCUMENT,"[The factual or reasoning answer appears here.]\n\nProvenance: LEARNED_STUDY\nCitation: (Sam-Wobo et al., 2013)\nStudy basis: [The semantic study descriptor appears here.]"
1,W114726911,VERIFIED_DOCUMENT,"[The factual or reasoning answer appears here.]\n\nProvenance: LEARNED_STUDY\nCitation: (Akpan et al., 2014)\nStudy basis: [The semantic study descriptor appears here.]"
2,W1263080160,CONFLICT,"[The factual or reasoning answer appears here.]\n\nProvenance: UNVERIFIED_STUDY\nCitation: (Nganje et al., 2015) [unverified]\nStudy basis: [The semantic study descriptor appears here.]"
3,W1426515992,VERIFIED_DOCUMENT,"[The factual or reasoning answer appears here.]\n\nProvenance: LEARNED_STUDY\nCitation: (Oyediran & Fadamoro, 2015)\nStudy basis: [The semantic study descriptor appears here.]"
4,W1480464361,VERIFIED_DOCUMENT,"[The factual or reasoning answer appears here.]\n\nProvenance: LEARNED_STUDY\nCitation: (Adebawo et al., 2006)\nStudy basis: [The semantic study descriptor appears here.]"
5,W1481448257,CONFLICT,"[The factual or reasoning answer appears here.]\n\nProvenance: UNVERIFIED_STUDY\nCitation: (Adesuyi et al., 2015) [unverified]\nStudy basis: [The semantic study descriptor appears here.]"
6,W1483996455,CONFLICT,"[The factual or reasoning answer appears here.]\n\nProvenance: UNVERIFIED_STUDY\nCitation: (Oladimeji et al., 2015) [unverified]\nStudy basis: [The semantic study descriptor appears here.]"
7,W1486712254,VERIFIED_DOCUMENT,"[The factual or reasoning answer appears here.]\n\nProvenance: LEARNED_STUDY\nCitation: (Adeniyan et al., 2007)\nStudy basis: [The semantic study descriptor appears here.]"
8,W1490899892,VERIFIED_DOCUMENT,"[The factual or reasoning answer appears here.]\n\nProvenance: LEARNED_STUDY\nCitation: (Ladokun & Oni, 2015)\nStudy basis: [The semantic study descriptor appears here.]"
9,W1493677654,CONFLICT,"[The factual or reasoning answer appears here.]\n\nProvenance: UNVERIFIED_STUDY\nCitation: (Akande, 2015) [unverified]\nStudy basis: [The semantic study descriptor appears here.]"


Preview complete. This notebook applied no training-data gate.
To prepare the canonical table, review these cells, set FULL_RUN=True, and Run all.
